In [0]:
from pyspark.sql import SparkSession

# 1. Configurar a sessão (caso não esteja pré-configurada no teu cluster)
# O conector do BigQuery lida automaticamente com a autenticação via GCP (Google Cloud)
spark = SparkSession.builder \
    .appName("Databricks BigQuery Integration") \
    .getOrCreate()

# Definir o projeto e a tabela do BigQuery de origem
project_id = "o-teu-projeto-gcp"
dataset_id = "o_teu_dataset"
table_id = "a_tua_tabela"
full_table_name = f"{project_id}.{dataset_id}.{table_id}"

# -------------------------------------------------------------
# 2. LOAD / READ: Carregar dados do BigQuery para um DataFrame Spark
# -------------------------------------------------------------
print("A carregar dados do BigQuery...")
df_bq = spark.read \
    .format("bigquery") \
    .option("table", full_table_name) \
    .load()

# Exibir os primeiros registos no Databricks
display(df_bq.limit(5))


# -------------------------------------------------------------
# 3. QUERY: Executar uma consulta SQL diretamente no BigQuery (Pushdown)
# -------------------------------------------------------------
# Se quiseres filtrar os dados diretamente no BigQuery antes de trazer para o Spark
query_sql = f"""
    SELECT user_id, signup_date, country 
    FROM `{full_table_name}` 
    WHERE active = true
"""

df_filtered = spark.read \
    .format("bigquery") \
    .option("query", query_sql) \
    .load()


# -------------------------------------------------------------
# 4. COLLECT(): Trazer os resultados agregados/filtrados para o Driver
# -------------------------------------------------------------
# ATENÇÃO: O uso de collect() traz os dados da memória distribuída do Spark 
# para o nó Driver. Utiliza apenas em datasets pequenos ou após agregações.
print("A recolher resultados para o Driver com collect()...")

# Exemplo: Agrupar por país e recolher a lista de linhas para o Python local
df_agg = df_filtered.groupBy("country").count()

# collect() retorna uma lista de objetos Row do PySpark
collected_rows = df_agg.collect()

print("\nResultados recolhidos no Driver:")
for row in collected_rows:
    print(f"País: {row['country']} | Contagem: {row['count']}")